# TimeGAN baseline aligned with Conv1D Hybrid Detector

In [1]:
import os
import json
import math
import joblib
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

c:\Users\j.manriquec\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reproducibilidad y dispositivo

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Cargar configuración del mejor detector

In [3]:
PROJECT_ROOT = Path("..").resolve()

detector_config_path = PROJECT_ROOT / "models" / "detectors" / "conv1d_hybrid_ae" / "config.json"

with open(detector_config_path, "r") as f:
    detector_config = json.load(f)

detector_config

{'model_name': 'conv1d_hybrid_ae',
 'best_params_from_optuna': {'window_size': 60,
  'stride': 1,
  'n_conv_layers': 1,
  'conv_channels_1': 64,
  'conv_channels_2': 16,
  'conv_channels_3': 8,
  'kernel_size': 5,
  'disc_hidden_1': 64,
  'disc_hidden_2': 16,
  'fusion_hidden': 256,
  'dropout': 0.01837443752413843,
  'batch_size': 128,
  'lr': 0.00023729585064585673,
  'optimizer': 'Adam',
  'epochs': 30},
 'continuous_cols': ['AIT201.Pv',
  'AIT202.Pv',
  'AIT203.Pv',
  'AIT501.Pv',
  'AIT502.Pv',
  'AIT503.Pv',
  'AIT504.Pv',
  'DPIT301.Pv',
  'FIT101.Pv',
  'FIT201.Pv',
  'FIT301.Pv',
  'FIT401.Pv',
  'FIT501.Pv',
  'FIT502.Pv',
  'FIT503.Pv',
  'FIT504.Pv',
  'FIT601.Pv',
  'LIT101.Pv',
  'LIT301.Pv',
  'LIT401.Pv',
  'PIT501.Pv',
  'PIT502.Pv',
  'PIT503.Pv'],
 'discrete_cols': ['MV101.Status',
  'MV201.Status',
  'MV301.Status',
  'MV302.Status',
  'MV303.Status',
  'MV304.Status',
  'MV501.Status',
  'MV502.Status',
  'MV503.Status',
  'MV504.Status',
  'P101.Status',
  'P201.S

In [4]:
continuous_cols = detector_config["continuous_cols"]
discrete_cols = detector_config["discrete_cols"]
feature_cols = detector_config["feature_cols"]

WINDOW_SIZE = detector_config["window_size"]   # debería ser 60
STRIDE = detector_config["stride"]             # debería ser 1
WARMUP = detector_config["warmup"]

print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Warmup:", WARMUP)
print("Continuous:", len(continuous_cols))
print("Discrete:", len(discrete_cols))

Window size: 60
Stride: 1
Warmup: 120
Continuous: 23
Discrete: 26


## Cargar datos limpios

In [5]:
clean_folder = PROJECT_ROOT / "data" / "clean" / "SWaT.A7_June2020_clean"
file_paths = sorted(clean_folder.glob("*.csv"))

def load_clean_csv(path):
    df = pd.read_csv(path)
    df["t_stamp"] = pd.to_datetime(df["t_stamp"], errors="coerce")
    df = df.sort_values("t_stamp").reset_index(drop=True)
    return df

files = {fp.stem: load_clean_csv(fp) for fp in file_paths}

print("Archivos cargados:")
list(files.keys())

Archivos cargados:


['22June2020_1', '22June2020_2', '29June2020_1', '29June2020_2']

## Columnas comunes

In [6]:
common_cols = sorted(set.intersection(*(set(df.columns) for df in files.values())))
files = {name: df[common_cols].copy() for name, df in files.items()}

print("Número de columnas comunes:", len(common_cols))

Número de columnas comunes: 61


## Eliminar filas completamente vacías

In [7]:
temp_feature_cols = [c for c in next(iter(files.values())).columns if c != "t_stamp"]

for name in files:
    before = len(files[name])
    files[name] = files[name].dropna(subset=temp_feature_cols, how="all").reset_index(drop=True)
    after = len(files[name])

    if before != after:
        print(f"{name}: se eliminaron {before - after} filas completamente vacías")

29June2020_2: se eliminaron 1 filas completamente vacías


## Eliminar columnas constantes

In [8]:
profile_rows = []

global_df = pd.concat([df[feature_cols] for df in files.values()], axis=0).reset_index(drop=True)

for col in feature_cols:
    s = global_df[col]
    profile_rows.append({
        "feature": col,
        "type": "continuous" if col in continuous_cols else "discrete",
        "n_unique": s.nunique(dropna=True),
        "missing": int(s.isna().sum()),
        "is_constant": s.nunique(dropna=True) <= 1
    })

profile_df = pd.DataFrame(profile_rows)
constant_cols = profile_df.loc[profile_df["is_constant"], "feature"].tolist()

constant_cols

[]

In [9]:
for name in files:
    files[name] = files[name].drop(columns=constant_cols, errors="ignore")

continuous_cols = [c for c in continuous_cols if c not in constant_cols]
discrete_cols = [c for c in discrete_cols if c not in constant_cols]
feature_cols = continuous_cols + discrete_cols

print("Continuas restantes:", len(continuous_cols))
print("Discretas restantes:", len(discrete_cols))

Continuas restantes: 23
Discretas restantes: 26


## Warm-up

In [10]:
def remove_warmup(df, warmup_seconds=120):
    if "t_stamp" not in df.columns:
        return df.copy()
    start_time = df["t_stamp"].min()
    return df[df["t_stamp"] >= start_time + pd.Timedelta(seconds=warmup_seconds)].reset_index(drop=True)

for name in files:
    files[name] = remove_warmup(files[name], warmup_seconds=WARMUP)

## Split temporal por archivo

In [11]:
def temporal_split(df, train_ratio=0.6, val_ratio=0.2):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    return train_df, val_df, test_df

In [12]:
split_by_file = {}

for name, df_i in files.items():
    df_base = df_i[["t_stamp"] + feature_cols].copy()

    train_i, val_i, test_i = temporal_split(df_base)

    split_by_file[name] = {
        "train": train_i,
        "val": val_i,
        "test": test_i
    }

    print(f"{name}:")
    print("  train:", train_i.shape)
    print("  val:  ", val_i.shape)
    print("  test: ", test_i.shape)

22June2020_1:
  train: (8568, 50)
  val:   (2856, 50)
  test:  (2856, 50)
22June2020_2:
  train: (2088, 50)
  val:   (696, 50)
  test:  (696, 50)
29June2020_1:
  train: (4248, 50)
  val:   (1416, 50)
  test:  (1417, 50)
29June2020_2:
  train: (4248, 50)
  val:   (1416, 50)
  test:  (1416, 50)


## Escalado de continuas

In [13]:
# ---------------------------------
# Escaladores:
# 1) scaler_cont_detector: mantiene compatibilidad con el detector final
# 2) scaler_cont_timegan: escala continuas a [0,1] para entrenar TimeGAN
# ---------------------------------

scaler_cont_detector = StandardScaler()
scaler_cont_timegan = MinMaxScaler()

train_all_cont = pd.concat(
    [split_by_file[name]["train"][continuous_cols] for name in split_by_file],
    axis=0
)

# Escalador del detector (como ya lo venías usando en detectores)
scaler_cont_detector.fit(train_all_cont)

# Escalador específico para TimeGAN
scaler_cont_timegan.fit(train_all_cont)

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


In [14]:
def scale_split_part(df_part, scaler_cont_timegan, continuous_cols, discrete_cols):
    """
    Para TimeGAN:
    - continuas -> MinMaxScaler [0,1]
    - discretas -> se dejan crudas para luego hacer one-hot por timestep
    """
    cont = pd.DataFrame(
        scaler_cont_timegan.transform(df_part[continuous_cols]),
        columns=continuous_cols,
        index=df_part.index
    )

    disc = df_part[discrete_cols].copy()

    return cont, disc

In [15]:
scaled_split_by_file = {}

for name in split_by_file:
    scaled_split_by_file[name] = {}

    for subset in ["train", "val", "test"]:
        df_part = split_by_file[name][subset]

        cont_part, disc_part = scale_split_part(
            df_part=df_part,
            scaler_cont_timegan=scaler_cont_timegan,
            continuous_cols=continuous_cols,
            discrete_cols=discrete_cols
        )

        scaled_split_by_file[name][subset] = {
            "cont": cont_part,
            "disc": disc_part
        }

## Catálogo de categorías discretas usando train

In [16]:
discrete_value_maps = {}

train_disc_global = pd.concat(
    [scaled_split_by_file[name]["train"]["disc"] for name in scaled_split_by_file],
    axis=0
).reset_index(drop=True)

for col in discrete_cols:
    vals = sorted(pd.Series(train_disc_global[col]).dropna().unique().tolist())
    discrete_value_maps[col] = vals

discrete_value_maps

{'MV101.Status': [0.0, 1.0, 2.0],
 'MV201.Status': [0.0, 1.0, 2.0],
 'MV301.Status': [0.0, 1.0, 2.0],
 'MV302.Status': [0.0, 1.0, 2.0],
 'MV303.Status': [0.0, 1.0, 2.0],
 'MV304.Status': [0.0, 1.0, 2.0],
 'MV501.Status': [0.0, 1.0, 2.0],
 'MV502.Status': [0.0, 1.0, 2.0],
 'MV503.Status': [0.0, 1.0, 2.0],
 'MV504.Status': [0.0, 1.0, 2.0],
 'P101.Status': [1.0, 2.0],
 'P201.Status': [1.0, 2.0],
 'P203.Status': [1.0, 2.0],
 'P205.Status': [1.0, 2.0],
 'P301.Status': [1.0, 2.0],
 'P401.Status': [1.0, 2.0],
 'P501.Status': [1.0, 2.0],
 'P601.Status': [1.0, 2.0],
 'P602.Status': [1.0, 2.0],
 'UV401.Status': [1.0, 2.0],
 'P1_STATE': [1.0, 2.0, 3.0],
 'P2_STATE': [1.0, 2.0],
 'P3_STATE': [1.0,
  2.0,
  4.0,
  5.0,
  6.0,
  7.0,
  9.0,
  10.0,
  12.0,
  13.0,
  14.0,
  15.0,
  16.0,
  99.0],
 'P4_STATE': [1.0, 2.0, 3.0, 4.0],
 'P5_STATE': [1.0,
  3.0,
  4.0,
  5.0,
  6.0,
  8.0,
  9.0,
  10.0,
  11.0,
  12.0,
  15.0,
  16.0,
  17.0,
  18.0,
  19.0,
  21.0],
 'P6_STATE': [1.0, 2.0]}

In [17]:
onehot_sizes = {col: len(discrete_value_maps[col]) for col in discrete_cols}
total_disc_onehot_dim = sum(onehot_sizes.values())

print("Total one-hot dim discretas:", total_disc_onehot_dim)

Total one-hot dim discretas: 91


## One-hot encoding por timestep

In [18]:
def one_hot_encode_discrete_sequence(disc_array, discrete_cols, discrete_value_maps):
    """
    disc_array: np.array con shape (L, n_discrete)
    retorna: np.array con shape (L, total_onehot_dim)
    """
    rows = []

    for t in range(disc_array.shape[0]):
        encoded_t = []

        for j, col in enumerate(discrete_cols):
            value = disc_array[t, j]
            categories = discrete_value_maps[col]

            onehot = np.zeros(len(categories), dtype=np.float32)

            if value in categories:
                idx = categories.index(value)
                onehot[idx] = 1.0

            encoded_t.extend(onehot.tolist())

        rows.append(encoded_t)

    return np.array(rows, dtype=np.float32)

## Construir ventanas por archivo

In [19]:
def build_windows_for_timegan(
    scaled_split_by_file,
    subset,
    continuous_cols,
    discrete_cols,
    discrete_value_maps,
    window_size=60,
    stride=1
):
    cont_windows = []
    disc_windows_raw = []
    disc_windows_onehot = []
    origin_tracker = []

    for name in scaled_split_by_file:
        cont_df = scaled_split_by_file[name][subset]["cont"]
        disc_df = scaled_split_by_file[name][subset]["disc"]

        if len(cont_df) < window_size:
            continue

        cont_arr = cont_df.values.astype(np.float32)
        disc_arr = disc_df.values

        for i in range(0, len(cont_df) - window_size + 1, stride):
            cont_w = cont_arr[i:i+window_size]
            disc_w = disc_arr[i:i+window_size]

            disc_w_onehot = one_hot_encode_discrete_sequence(
                disc_w, discrete_cols, discrete_value_maps
            )

            cont_windows.append(cont_w)
            disc_windows_raw.append(disc_w)
            disc_windows_onehot.append(disc_w_onehot)
            origin_tracker.append(name)

    if len(cont_windows) == 0:
        return None

    return (
        np.array(cont_windows, dtype=np.float32),
        np.array(disc_windows_raw, dtype=object),
        np.array(disc_windows_onehot, dtype=np.float32),
        origin_tracker
    )

## Construir ventanas train/val/test

In [20]:
train_pack = build_windows_for_timegan(
    scaled_split_by_file=scaled_split_by_file,
    subset="train",
    continuous_cols=continuous_cols,
    discrete_cols=discrete_cols,
    discrete_value_maps=discrete_value_maps,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

val_pack = build_windows_for_timegan(
    scaled_split_by_file=scaled_split_by_file,
    subset="val",
    continuous_cols=continuous_cols,
    discrete_cols=discrete_cols,
    discrete_value_maps=discrete_value_maps,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

test_pack = build_windows_for_timegan(
    scaled_split_by_file=scaled_split_by_file,
    subset="test",
    continuous_cols=continuous_cols,
    discrete_cols=discrete_cols,
    discrete_value_maps=discrete_value_maps,
    window_size=WINDOW_SIZE,
    stride=STRIDE
)

In [21]:
train_cont_windows, train_disc_windows_raw, train_disc_windows_onehot, train_origin = train_pack
val_cont_windows, val_disc_windows_raw, val_disc_windows_onehot, val_origin = val_pack
test_cont_windows, test_disc_windows_raw, test_disc_windows_onehot, test_origin = test_pack

print("Train cont windows:", train_cont_windows.shape)
print("Train disc raw windows:", train_disc_windows_raw.shape)
print("Train disc onehot windows:", train_disc_windows_onehot.shape)

print("Val cont windows:", val_cont_windows.shape)
print("Val disc onehot windows:", val_disc_windows_onehot.shape)

print("Test cont windows:", test_cont_windows.shape)
print("Test disc onehot windows:", test_disc_windows_onehot.shape)

Train cont windows: (18916, 60, 23)
Train disc raw windows: (18916, 60, 26)
Train disc onehot windows: (18916, 60, 91)
Val cont windows: (6148, 60, 23)
Val disc onehot windows: (6148, 60, 91)
Test cont windows: (6149, 60, 23)
Test disc onehot windows: (6149, 60, 91)


## Dataset final para TimeGAN

In [22]:
train_timegan_data = np.concatenate(
    [train_cont_windows, train_disc_windows_onehot],
    axis=2
)

val_timegan_data = np.concatenate(
    [val_cont_windows, val_disc_windows_onehot],
    axis=2
)

test_timegan_data = np.concatenate(
    [test_cont_windows, test_disc_windows_onehot],
    axis=2
)

print("Train TimeGAN data:", train_timegan_data.shape)
print("Val TimeGAN data:", val_timegan_data.shape)
print("Test TimeGAN data:", test_timegan_data.shape)

Train TimeGAN data: (18916, 60, 114)
Val TimeGAN data: (6148, 60, 114)
Test TimeGAN data: (6149, 60, 114)


## Estadísticas de control de continuas reales

In [23]:
print("Rango continuas TRAIN para TimeGAN:")
print("Min:", train_cont_windows.min())
print("Max:", train_cont_windows.max())

Rango continuas TRAIN para TimeGAN:
Min: 0.0
Max: 1.0


## Dataloader de train para TimeGAN

In [24]:
BATCH_SIZE = 64

train_timegan_tensor = torch.tensor(train_timegan_data, dtype=torch.float32)
train_loader = DataLoader(
    TensorDataset(train_timegan_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True
)

## Parámetros base de TimeGAN

In [25]:
seq_len = WINDOW_SIZE
feature_dim = train_timegan_data.shape[2]

hidden_dim = 64
latent_dim = 64
num_layers = 3

epochs_embedder = 50
epochs_supervisor = 50
epochs_joint = 100

gamma = 1
eta = 200      # peso de la pérdida supervisada en generador/supervisor
lambda_r = 10  # peso de reconstrucción del embedder
lambda_s = 1.0 # peso de la pérdida supervisada en embedder
lambda_m = 100 # peso de moment matching

## Componentes de TimeGAN

In [26]:
class Embedder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.rnn = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.Sigmoid()

    def forward(self, x):
        h, _ = self.rnn(x)
        return self.act(self.fc(h))

In [27]:
class Recovery(nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.rnn = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.act = nn.Sigmoid()

    def forward(self, h):
        x_tilde, _ = self.rnn(h)
        return self.act(self.fc(x_tilde))

In [28]:
class Generator(nn.Module):
    def __init__(self, z_dim, hidden_dim, num_layers):
        super().__init__()
        self.rnn = nn.GRU(z_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.Sigmoid()

    def forward(self, z):
        e_hat, _ = self.rnn(z)
        return self.act(self.fc(e_hat))

In [29]:
class Supervisor(nn.Module):
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.rnn = nn.GRU(hidden_dim, hidden_dim, num_layers=max(1, num_layers - 1), batch_first=True)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.Sigmoid()

    def forward(self, h):
        s, _ = self.rnn(h)
        return self.act(self.fc(s))

In [30]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.rnn = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        y_hat, _ = self.rnn(h)
        return self.fc(y_hat)

## Inicializar redes

In [31]:
embedder = Embedder(feature_dim, hidden_dim, num_layers).to(device)
recovery = Recovery(hidden_dim, feature_dim, num_layers).to(device)
generator = Generator(latent_dim, hidden_dim, num_layers).to(device)
supervisor = Supervisor(hidden_dim, num_layers).to(device)
discriminator = Discriminator(hidden_dim, num_layers).to(device)

## Losses y optimizadores

In [32]:
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

e_optimizer = torch.optim.Adam(list(embedder.parameters()) + list(recovery.parameters()), lr=1e-3)
s_optimizer = torch.optim.Adam(supervisor.parameters(), lr=1e-3)
g_optimizer = torch.optim.Adam(list(generator.parameters()) + list(supervisor.parameters()), lr=5e-4)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4)

## Funciones auxiliares

In [33]:
def random_generator(batch_size, seq_len, z_dim, device):
    return torch.rand(batch_size, seq_len, z_dim, device=device)

In [34]:
def moment_loss(y_true, y_pred):
    mean_loss = torch.mean(torch.abs(torch.mean(y_true, dim=0) - torch.mean(y_pred, dim=0)))
    var_loss = torch.mean(torch.abs(torch.var(y_true, dim=0) - torch.var(y_pred, dim=0)))
    return mean_loss + var_loss

## Decodificar discretas one-hot a secuencia cruda

In [35]:
def decode_onehot_discrete_sequence(onehot_seq, discrete_cols, discrete_value_maps):
    """
    onehot_seq: (L, total_onehot_dim)
    retorna: (L, n_discrete)
    """
    decoded_rows = []

    for t in range(onehot_seq.shape[0]):
        row = []
        cursor = 0

        for col in discrete_cols:
            categories = discrete_value_maps[col]
            k = len(categories)

            block = onehot_seq[t, cursor:cursor+k]
            idx = int(np.argmax(block))
            row.append(categories[idx])

            cursor += k

        decoded_rows.append(row)

    return np.array(decoded_rows, dtype=np.float32)

## Resumen de discretas sintéticas por ventana

In [36]:
def summarize_discrete_windows_from_raw(disc_windows_raw, discrete_cols):
    summary_rows = []

    for w in disc_windows_raw:
        feats = {}

        for j, col in enumerate(discrete_cols):
            values = w[:, j]

            mode_val = pd.Series(values).mode().iloc[0]
            transitions = np.sum(values[1:] != values[:-1])
            nunique = len(np.unique(values))
            dominance = pd.Series(values).value_counts(normalize=True).iloc[0]

            feats[f"{col}__mode"] = mode_val
            feats[f"{col}__transitions"] = transitions
            feats[f"{col}__nunique"] = nunique
            feats[f"{col}__dominance"] = dominance

        summary_rows.append(feats)

    return pd.DataFrame(summary_rows)

## Submuestra fija para evaluación rápida

In [37]:
EVAL_N = min(500, len(train_timegan_data))
eval_real_windows = train_timegan_data[:EVAL_N].copy()

print("Evaluation subset shape:", eval_real_windows.shape)

Evaluation subset shape: (500, 60, 114)


## Función para construir modelos según el trial

In [38]:
def build_timegan_modules(feature_dim, hidden_dim, latent_dim, num_layers, device):
    embedder = Embedder(feature_dim, hidden_dim, num_layers).to(device)
    recovery = Recovery(hidden_dim, feature_dim, num_layers).to(device)
    generator = Generator(latent_dim, hidden_dim, num_layers).to(device)
    supervisor = Supervisor(hidden_dim, num_layers).to(device)
    discriminator = Discriminator(hidden_dim, num_layers).to(device)

    return embedder, recovery, generator, supervisor, discriminator

## Métrica proxy de similitud real-sintético

In [39]:
def compute_similarity_score(
    real_windows,
    synthetic_windows,
    n_cont,
    discrete_cols,
    discrete_value_maps
):
    """
    Menor score = mejor
    real_windows, synthetic_windows: (N, L, D_total)
    """

    # -------------------------
    # Separar continuas y discretas one-hot
    # -------------------------
    real_cont = real_windows[:, :, :n_cont]
    synth_cont = synthetic_windows[:, :, :n_cont]

    real_disc_onehot = real_windows[:, :, n_cont:]
    synth_disc_onehot = synthetic_windows[:, :, n_cont:]

    # -------------------------
    # Score continuas: medias y std
    # -------------------------
    real_cont_flat = real_cont.reshape(-1, n_cont)
    synth_cont_flat = synth_cont.reshape(-1, n_cont)

    mean_gap = np.mean(np.abs(real_cont_flat.mean(axis=0) - synth_cont_flat.mean(axis=0)))
    std_gap = np.mean(np.abs(real_cont_flat.std(axis=0) - synth_cont_flat.std(axis=0)))

    score_cont = mean_gap + std_gap

    # -------------------------
    # Decodificar discretas sintéticas y reales
    # -------------------------
    real_disc_raw = []
    synth_disc_raw = []

    for i in range(len(real_windows)):
        real_decoded = decode_onehot_discrete_sequence(
            real_disc_onehot[i], discrete_cols, discrete_value_maps
        )
        synth_decoded = decode_onehot_discrete_sequence(
            synth_disc_onehot[i], discrete_cols, discrete_value_maps
        )

        real_disc_raw.append(real_decoded)
        synth_disc_raw.append(synth_decoded)

    real_disc_raw = np.array(real_disc_raw, dtype=np.float32)
    synth_disc_raw = np.array(synth_disc_raw, dtype=np.float32)

    real_disc_summary = summarize_discrete_windows_from_raw(real_disc_raw, discrete_cols)
    synth_disc_summary = summarize_discrete_windows_from_raw(synth_disc_raw, discrete_cols)

    common_disc_cols = [c for c in real_disc_summary.columns if c in synth_disc_summary.columns]

    disc_gap = np.mean(
        np.abs(
            real_disc_summary[common_disc_cols].mean(axis=0).values -
            synth_disc_summary[common_disc_cols].mean(axis=0).values
        )
    )

    # -------------------------
    # Score PCA: centroides
    # -------------------------
    n_pca = min(len(real_windows), len(synthetic_windows), 300)

    real_pca_input = real_windows[:n_pca].reshape(n_pca, -1)
    synth_pca_input = synthetic_windows[:n_pca].reshape(n_pca, -1)

    pca = PCA(n_components=2, random_state=SEED)
    real_pca = pca.fit_transform(real_pca_input)
    synth_pca = pca.transform(synth_pca_input)

    centroid_gap = np.linalg.norm(real_pca.mean(axis=0) - synth_pca.mean(axis=0))

    # -------------------------
    # Penalización por colapso
    # -------------------------
    synth_var = synth_cont_flat.var(axis=0).mean()
    collapse_penalty = 0.0 if synth_var > 1e-4 else 10.0

    score = score_cont + disc_gap + 0.5 * centroid_gap + collapse_penalty

    return float(score)

## Entrenamiento corto de un trial

In [40]:
def train_timegan_trial(
    train_loader,
    feature_dim,
    hidden_dim,
    latent_dim,
    num_layers,
    lr_e,
    lr_s,
    lr_g,
    lr_d,
    eta,
    lambda_r,
    lambda_s,
    lambda_m,
    gamma,
    epochs_embedder,
    epochs_supervisor,
    epochs_joint,
    device
):
    embedder, recovery, generator, supervisor, discriminator = build_timegan_modules(
        feature_dim=feature_dim,
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        num_layers=num_layers,
        device=device
    )

    mse_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    e_optimizer = torch.optim.Adam(
        list(embedder.parameters()) + list(recovery.parameters()),
        lr=lr_e
    )
    s_optimizer = torch.optim.Adam(supervisor.parameters(), lr=lr_s)
    g_optimizer = torch.optim.Adam(
        list(generator.parameters()) + list(supervisor.parameters()),
        lr=lr_g
    )
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=lr_d)

    seq_len = WINDOW_SIZE

    # -------------------------
    # Phase 1: Embedder
    # -------------------------
    for _ in range(epochs_embedder):
        for batch in train_loader:
            x = batch[0].to(device)

            e_optimizer.zero_grad()
            h = embedder(x)
            x_tilde = recovery(h)

            e_loss_t0 = mse_loss(x_tilde, x)
            e_loss = 10 * torch.sqrt(e_loss_t0 + 1e-8)

            e_loss.backward()
            e_optimizer.step()

    # -------------------------
    # Phase 2: Supervisor
    # -------------------------
    for _ in range(epochs_supervisor):
        for batch in train_loader:
            x = batch[0].to(device)

            s_optimizer.zero_grad()

            with torch.no_grad():
                h = embedder(x)

            h_hat_supervise = supervisor(h)
            s_loss = mse_loss(h[:, 1:, :], h_hat_supervise[:, :-1, :])

            s_loss.backward()
            s_optimizer.step()

    # -------------------------
    # Phase 3: Joint
    # -------------------------
    for _ in range(epochs_joint):
        for batch in train_loader:
            x = batch[0].to(device)
            batch_size_curr = x.shape[0]

            for _ in range(2):
                # Generator + Supervisor
                g_optimizer.zero_grad()

                h = embedder(x)
                z = random_generator(batch_size_curr, seq_len, latent_dim, device)
                e_hat = generator(z)
                h_hat = supervisor(e_hat)
                x_hat = recovery(h_hat)

                y_fake = discriminator(h_hat)
                y_fake_e = discriminator(e_hat)

                g_loss_u = bce_loss(y_fake, torch.ones_like(y_fake))
                g_loss_u_e = bce_loss(y_fake_e, torch.ones_like(y_fake_e))

                h_supervise = supervisor(h)
                g_loss_s = mse_loss(h[:, 1:, :], h_supervise[:, :-1, :])

                g_loss_v = moment_loss(x, x_hat)

                g_loss = (
                    g_loss_u
                    + gamma * g_loss_u_e
                    + eta * torch.sqrt(g_loss_s + 1e-8)
                    + lambda_m * g_loss_v
                )

                g_loss.backward()
                g_optimizer.step()

                # Embedder refinement
                e_optimizer.zero_grad()

                h_ref = embedder(x)
                x_tilde = recovery(h_ref)
                h_supervise_ref = supervisor(h_ref)

                g_loss_s_ref = mse_loss(h_ref[:, 1:, :], h_supervise_ref[:, :-1, :])
                e_loss_t0 = mse_loss(x_tilde, x)
                e_loss0 = 10 * torch.sqrt(e_loss_t0 + 1e-8)
                e_loss = lambda_r * e_loss0 + lambda_s * g_loss_s_ref

                e_loss.backward()
                e_optimizer.step()

            # Discriminator
            d_optimizer.zero_grad()

            with torch.no_grad():
                h = embedder(x)
                z = random_generator(batch_size_curr, seq_len, latent_dim, device)
                e_hat = generator(z)
                h_hat = supervisor(e_hat)

            y_real = discriminator(h)
            y_fake = discriminator(h_hat)
            y_fake_e = discriminator(e_hat)

            d_loss_real = bce_loss(y_real, torch.ones_like(y_real))
            d_loss_fake = bce_loss(y_fake, torch.zeros_like(y_fake))
            d_loss_fake_e = bce_loss(y_fake_e, torch.zeros_like(y_fake_e))

            d_loss = d_loss_real + d_loss_fake + gamma * d_loss_fake_e

            if d_loss.item() > 0.15:
                d_loss.backward()
                d_optimizer.step()

    return embedder, recovery, generator, supervisor, discriminator

## Generación sintética para evaluación

In [41]:
def generate_synthetic_windows(generator, supervisor, recovery, n_samples, seq_len, latent_dim, device):
    with torch.no_grad():
        z = random_generator(n_samples, seq_len, latent_dim, device)
        e_hat = generator(z)
        h_hat = supervisor(e_hat)
        synthetic_windows = recovery(h_hat).cpu().numpy()

    return synthetic_windows

## Objective de Optuna

In [42]:
def objective(trial):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # -------------------------
    # Hiperparámetros
    # -------------------------
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128])
    latent_dim = trial.suggest_categorical("latent_dim", [32, 64])
    num_layers = trial.suggest_categorical("num_layers", [2, 3])

    eta = trial.suggest_categorical("eta", [100, 200, 300])
    lambda_r = trial.suggest_categorical("lambda_r", [5, 10, 20])
    lambda_s = trial.suggest_categorical("lambda_s", [0.1, 1.0, 5.0])
    lambda_m = trial.suggest_categorical("lambda_m", [50, 100])

    lr_e = trial.suggest_categorical("lr_e", [1e-3])
    lr_s = trial.suggest_categorical("lr_s", [1e-3])
    lr_g = trial.suggest_categorical("lr_g", [1e-4, 5e-4])
    lr_d = trial.suggest_categorical("lr_d", [5e-5, 1e-4])

    gamma = trial.suggest_categorical("gamma", [1])
    epochs_embedder = trial.suggest_categorical("epochs_embedder", [30, 50])
    epochs_supervisor = trial.suggest_categorical("epochs_supervisor", [30, 50])
    epochs_joint = trial.suggest_categorical("epochs_joint", [100, 150])

    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    # -------------------------
    # DataLoader
    # -------------------------
    train_timegan_tensor = torch.tensor(train_timegan_data, dtype=torch.float32)
    train_loader = DataLoader(
        TensorDataset(train_timegan_tensor),
        batch_size=batch_size,
        shuffle=True
    )

    # -------------------------
    # Entrenar
    # -------------------------
    embedder, recovery, generator, supervisor, discriminator = train_timegan_trial(
        train_loader=train_loader,
        feature_dim=train_timegan_data.shape[2],
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        num_layers=num_layers,
        lr_e=lr_e,
        lr_s=lr_s,
        lr_g=lr_g,
        lr_d=lr_d,
        eta=eta,
        lambda_r=lambda_r,
        lambda_s=lambda_s,
        lambda_m=lambda_m,
        gamma=gamma,
        epochs_embedder=epochs_embedder,
        epochs_supervisor=epochs_supervisor,
        epochs_joint=epochs_joint,
        device=device
    )

    # -------------------------
    # Generar y evaluar
    # -------------------------
    synthetic_eval = generate_synthetic_windows(
        generator=generator,
        supervisor=supervisor,
        recovery=recovery,
        n_samples=len(eval_real_windows),
        seq_len=WINDOW_SIZE,
        latent_dim=latent_dim,
        device=device
    )

    score = compute_similarity_score(
        real_windows=eval_real_windows,
        synthetic_windows=synthetic_eval,
        n_cont=len(continuous_cols),
        discrete_cols=discrete_cols,
        discrete_value_maps=discrete_value_maps
    )

    return score

## Crear estudio y correr

In [43]:
study = optuna.create_study(direction="minimize", study_name="timegan_conv1d_hybrid_aligned")
study.optimize(objective, n_trials=20)

[I 2026-04-06 12:04:44,305] A new study created in memory with name: timegan_conv1d_hybrid_aligned
[I 2026-04-06 12:11:20,194] Trial 0 finished with value: 3.8030110297337556 and parameters: {'hidden_dim': 64, 'latent_dim': 32, 'num_layers': 2, 'eta': 200, 'lambda_r': 20, 'lambda_s': 1.0, 'lambda_m': 100, 'lr_e': 0.001, 'lr_s': 0.001, 'lr_g': 0.0001, 'lr_d': 0.0001, 'gamma': 1, 'epochs_embedder': 30, 'epochs_supervisor': 30, 'epochs_joint': 100, 'batch_size': 128}. Best is trial 0 with value: 3.8030110297337556.
[I 2026-04-06 12:18:31,910] Trial 1 finished with value: 2.850442569614068 and parameters: {'hidden_dim': 64, 'latent_dim': 64, 'num_layers': 2, 'eta': 100, 'lambda_r': 5, 'lambda_s': 5.0, 'lambda_m': 100, 'lr_e': 0.001, 'lr_s': 0.001, 'lr_g': 0.0005, 'lr_d': 5e-05, 'gamma': 1, 'epochs_embedder': 30, 'epochs_supervisor': 50, 'epochs_joint': 100, 'batch_size': 128}. Best is trial 1 with value: 2.850442569614068.
[I 2026-04-06 12:26:26,185] Trial 2 finished with value: 3.48855157

## Mejores resultados

In [44]:
print("Best trial:")
print("  Value:", study.best_trial.value)
print("  Params:")

for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Value: 2.2743748154738013
  Params:
    hidden_dim: 128
    latent_dim: 64
    num_layers: 3
    eta: 300
    lambda_r: 20
    lambda_s: 1.0
    lambda_m: 50
    lr_e: 0.001
    lr_s: 0.001
    lr_g: 0.0005
    lr_d: 0.0001
    gamma: 1
    epochs_embedder: 50
    epochs_supervisor: 50
    epochs_joint: 100
    batch_size: 128


In [45]:
optuna_results = study.trials_dataframe()
optuna_results.head()

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_epochs_embedder,params_epochs_joint,params_epochs_supervisor,params_eta,...,params_lambda_m,params_lambda_r,params_lambda_s,params_latent_dim,params_lr_d,params_lr_e,params_lr_g,params_lr_s,params_num_layers,state
0,0,3.803011,2026-04-06 12:04:44.306101,2026-04-06 12:11:20.194649,0 days 00:06:35.888548,128,30,100,30,200,...,100,20,1.0,32,0.00010,0.001,0.0001,0.001,2,COMPLETE
1,1,2.850443,2026-04-06 12:11:20.195651,2026-04-06 12:18:31.909971,0 days 00:07:11.714320,128,30,100,50,100,...,100,5,5.0,64,0.00005,0.001,0.0005,0.001,2,COMPLETE
2,2,3.488552,2026-04-06 12:18:31.910680,2026-04-06 12:26:26.185896,0 days 00:07:54.275216,128,50,100,50,200,...,50,10,5.0,32,0.00010,0.001,0.0001,0.001,3,COMPLETE
3,3,5.316959,2026-04-06 12:26:26.187724,2026-04-06 13:20:02.411168,0 days 00:53:36.223444,64,50,100,50,100,...,100,5,1.0,64,0.00010,0.001,0.0001,0.001,3,COMPLETE
4,4,3.060622,2026-04-06 13:20:02.412254,2026-04-06 14:51:41.181778,0 days 01:31:38.769524,64,30,150,50,200,...,50,20,1.0,32,0.00010,0.001,0.0001,0.001,2,COMPLETE
